# NER với PhoBERT cho truy vấn tìm kiếm thông minh

Mục tiêu: nhận diện thực thể kiểu (sản phẩm/dịch vụ) và thuộc tính (màu sắc, kích thước, tính năng, mùa, form, giới tính, giá) từ câu truy vấn.

- Input: CSV tại `/kaggle/input/...`
- Output: model/checkpoints tại `/kaggle/working/...`

Yêu cầu môi trường: `pandas`, `numpy`, `torch`, `transformers`, `seqeval`, `datasets`, `accelerate`.


In [ ]:
# Cài đặt thư viện (bỏ qua nếu môi trường đã có sẵn)
%pip -q install transformers==4.44.2 datasets==2.21.0 seqeval==1.2.2 accelerate==0.34.2 underthesea==6.8.4

import os
import re
import json
import math
import random
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd

import torch
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel, Sequence
from transformers import (AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification, TrainingArguments, Trainer)

SEED = 2024
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Đường dẫn Kaggle (dùng trực tiếp, không cần biến)
os.makedirs("/kaggle/working/phobert-ner", exist_ok=True)

# Cột trong CSV: xác định thực thể cần gán nhãn
TEXT_COL = "text"
ENTITY_COLS = [
    ("category", "TYPE"),  # kiểu sản phẩm/dịch vụ
    ("gender", "ATTR"),    # giới tính
    ("sleeve", "ATTR"),    # tay áo
    ("season", "ATTR"),    # mùa
    ("fit", "ATTR"),       # form dáng
    ("color", "ATTR"),     # màu sắc/hoa văn
    ("price", "ATTR"),     # giá
]



# Tiền xử lý: chuẩn hoá khoảng trắng
_whitespace_re = re.compile(r"\s+")

def normalize_space(text: str) -> str:
    return _whitespace_re.sub(" ", text).strip()

# Đọc CSV từ Kaggle input
train_df = pd.read_csv("/kaggle/input/abc/train.csv")
val_df = pd.read_csv("/kaggle/input/abc/val.csv")
test_df = pd.read_csv("/kaggle/input/abc/test.csv")

for df in (train_df, val_df, test_df):
    df[TEXT_COL] = df[TEXT_COL].astype(str).map(normalize_space)
    for col, _ in ENTITY_COLS:
        df[col] = df[col].astype(str).map(normalize_space)

print("Số dòng:", len(train_df), len(val_df), len(test_df))
train_df.head(2)


In [ ]:
# Sinh nhãn BIO từ các cột thuộc tính
from dataclasses import dataclass

LABELS = ["O", "B-TYPE", "I-TYPE", "B-ATTR", "I-ATTR"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

@dataclass
class Span:
    start: int
    end: int  # exclusive
    kind: str  # "TYPE" hoặc "ATTR"

# Tìm tất cả vị trí xuất hiện (case-insensitive), cho phép chồng lấn nhẹ
def find_spans(text: str, phrase: str, kind: str) -> List[Span]:
    if not phrase or phrase.lower() in {"nan", "none"}:
        return []
    t = text
    p = phrase.strip()
    if not p:
        return []
    # escape regex và tìm theo ranh giới tương đối linh hoạt
    pat = re.compile(re.escape(p), flags=re.IGNORECASE)
    spans: List[Span] = [Span(m.start(), m.end(), kind) for m in pat.finditer(t)]
    return spans

# Hợp nhất và ưu tiên TYPE > ATTR khi chồng lấn
def merge_spans(spans: List[Span]) -> List[Span]:
    if not spans:
        return []
    # Sắp theo (start asc, TYPE trước ATTR)
    def pri(s: Span) -> Tuple[int, int]:
        return (s.start, 0 if s.kind == "TYPE" else 1)
    spans = sorted(spans, key=pri)
    merged: List[Span] = []
    for s in spans:
        if not merged:
            merged.append(s)
            continue
        last = merged[-1]
        if s.start <= last.end:  # chồng lấn
            # ưu tiên TYPE, hoặc đoạn dài hơn
            if (s.kind == "TYPE" and last.kind != "TYPE") or (s.end - s.start) > (last.end - last.start):
                merged[-1] = s
            # nếu không, bỏ s
        else:
            merged.append(s)
    return merged

# Chuyển char-spans sang nhãn theo từ (word-level) dựa trên tách khoảng trắng
def words_and_offsets(text: str) -> Tuple[List[str], List[Tuple[int,int]]]:
    words: List[str] = []
    offsets: List[Tuple[int,int]] = []
    i = 0
    for part in re.finditer(r"\S+", text):
        start, end = part.start(), part.end()
        words.append(text[start:end])
        offsets.append((start, end))
    return words, offsets

def bio_for_text(text: str, row: pd.Series) -> Tuple[List[str], List[str]]:
    spans: List[Span] = []
    # category -> TYPE
    cat = str(row.get("category", ""))
    spans += find_spans(text, cat, "TYPE")
    # các thuộc tính -> ATTR
    for col, kind in ENTITY_COLS:
        if kind != "ATTR":
            continue
        val = str(row.get(col, ""))
        spans += find_spans(text, val, "ATTR")
    spans = merge_spans(spans)
    words, offsets = words_and_offsets(text)
    tags = ["O"] * len(words)
    for sp in spans:
        # gán nhãn cho các từ giao với span
        first_hit = True
        for idx, (ws, we) in enumerate(offsets):
            if we <= sp.start or ws >= sp.end:
                continue
            if first_hit:
                tags[idx] = f"B-{sp.kind}"
                first_hit = False
            else:
                # chỉ ghi đè O -> I-*, giữ B-* nếu đã là đầu của span khác
                if tags[idx] == "O":
                    tags[idx] = f"I-{sp.kind}"
    return words, tags

# Áp dụng cho train/val/test

def build_examples(df: pd.DataFrame) -> List[Dict[str, object]]:
    examples: List[Dict[str, object]] = []
    for _, row in df.iterrows():
        text = row[TEXT_COL]
        words, tags = bio_for_text(text, row)
        examples.append({"words": words, "ner_tags": tags, "text": text})
    return examples

train_examples = build_examples(train_df)
val_examples = build_examples(val_df)
test_examples = build_examples(test_df)

print(train_examples[0]["words"][:20])
print(train_examples[0]["ner_tags"][:20])


In [ ]:
# Tokenize và căn chỉnh nhãn theo subword của PhoBERT
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel, Sequence
from typing import List, Dict
import numpy as np

# Định nghĩa dự phòng nhãn nếu chưa có từ cell trước
if 'LABELS' not in globals():
    LABELS = ["O", "B-TYPE", "I-TYPE", "B-ATTR", "I-ATTR"]
if 'label2id' not in globals():
    label2id = {l: i for i, l in enumerate(LABELS)}
if 'id2label' not in globals():
    id2label = {i: l for l, i in label2id.items()}

MODEL_NAME = "vinai/phobert-base"

# Nếu chưa có train_examples/val_examples/test_examples thì tự tạo từ CSV
if 'train_examples' not in globals() or 'val_examples' not in globals() or 'test_examples' not in globals():
    import pandas as pd
    import re
    TEXT_COL = 'text'
    ENTITY_COLS = [
        ('category', 'TYPE'),
        ('gender', 'ATTR'),
        ('sleeve', 'ATTR'),
        ('season', 'ATTR'),
        ('fit', 'ATTR'),
        ('color', 'ATTR'),
        ('price', 'ATTR'),
    ]
    _whitespace_re = re.compile(r"\s+")
    def normalize_space(text: str) -> str:
        return _whitespace_re.sub(" ", str(text)).strip()

    from dataclasses import dataclass
    from typing import Tuple
    @dataclass
    class Span:
        start: int
        end: int
        kind: str
    def find_spans(text: str, phrase: str, kind: str):
        if not phrase or str(phrase).lower() in {"nan", "none"}:
            return []
        pat = re.compile(re.escape(str(phrase).strip()), flags=re.IGNORECASE)
        return [Span(m.start(), m.end(), kind) for m in pat.finditer(text)]
    def merge_spans(spans):
        if not spans:
            return []
        spans = sorted(spans, key=lambda s: (s.start, 0 if s.kind == 'TYPE' else 1))
        out = []
        for s in spans:
            if not out:
                out.append(s); continue
            last = out[-1]
            if s.start <= last.end:
                if (s.kind == 'TYPE' and last.kind != 'TYPE') or (s.end - s.start) > (last.end - last.start):
                    out[-1] = s
            else:
                out.append(s)
        return out
    def words_and_offsets(text: str):
        words, offsets = [], []
        for m in re.finditer(r"\S+", text):
            words.append(text[m.start():m.end()])
            offsets.append((m.start(), m.end()))
        return words, offsets
    def bio_for_text(text: str, row):
        spans = []
        spans += find_spans(text, row.get('category', ''), 'TYPE')
        for col, kind in ENTITY_COLS:
            if kind != 'ATTR':
                continue
            spans += find_spans(text, row.get(col, ''), 'ATTR')
        spans = merge_spans(spans)
        words, offsets = words_and_offsets(text)
        tags = ['O'] * len(words)
        for sp in spans:
            first = True
            for idx, (ws, we) in enumerate(offsets):
                if we <= sp.start or ws >= sp.end:
                    continue
                if first:
                    tags[idx] = f'B-{sp.kind}'; first = False
                else:
                    if tags[idx] == 'O':
                        tags[idx] = f'I-{sp.kind}'
        return words, tags
    def build_examples(df: pd.DataFrame):
        ex = []
        for _, row in df.iterrows():
            text = normalize_space(row[TEXT_COL])
            words, tags = bio_for_text(text, row)
            ex.append({'words': words, 'ner_tags': tags, 'text': text})
        return ex
    train_df = pd.read_csv('/kaggle/input/datasetabc/abc/train.csv')
    val_df = pd.read_csv('/kaggle/input/datasetabc/abc/val.csv')
    test_df = pd.read_csv('/kaggle/input/datasetabc/abc/test.csv')
    for df in (train_df, val_df, test_df):
        df[TEXT_COL] = df[TEXT_COL].map(normalize_space)
        for c, _k in ENTITY_COLS:
            df[c] = df[c].map(normalize_space)
    train_examples = build_examples(train_df)
    val_examples = build_examples(val_df)
    test_examples = build_examples(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# HuggingFace Datasets
features = Features({
    "words": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=LABELS)),
    "text": Value("string"),
})

def to_hf_dataset(examples: List[Dict[str, object]]) -> Dataset:
    return Dataset.from_list(examples, features=features)

raw_ds = DatasetDict({
    "train": to_hf_dataset(train_examples),
    "validation": to_hf_dataset(val_examples),
    "test": to_hf_dataset(test_examples),
})

# Map -> token-level labels (không phụ thuộc word_ids của fast tokenizer)

def tokenize_and_align_labels(example):
    words = example["words"]
    tags = example["ner_tags"]
    # 1) Tokenize theo từng từ để biết số subword
    pieces = []
    expanded_labels = []
    for i, (w, tag) in enumerate(zip(words, tags)):
        # PhoBERT (RoBERTa) slow tokenizer không nhận add_prefix_space; dùng tiền tố khoảng trắng thủ công
        sub_tokens = tokenizer.tokenize((" " if i != 0 else "") + w)
        if not sub_tokens:
            continue
        pieces.extend(sub_tokens)
        # Cho phép tag là int (ClassLabel) hoặc str
        tag_str = id2label[tag] if isinstance(tag, int) else tag
        expanded_labels.append(label2id[tag_str])  # B-*/I-* đã có sẵn từ bước BIO word-level
        # Với các sub-token sau của cùng một từ: chuyển B-* -> I-*
        tail_label = expanded_labels[-1]
        if tail_label in (label2id.get("B-TYPE"), label2id.get("B-ATTR")):
            i_label = label2id["I-" + id2label[tail_label][2:]]
        else:
            i_label = tail_label
        for _ in range(len(sub_tokens) - 1):
            expanded_labels.append(i_label)
    # 2) Chuyển sang ids và thêm special tokens
    input_ids = tokenizer.convert_tokens_to_ids(pieces)
    enc = tokenizer.prepare_for_model(
        input_ids,
        return_attention_mask=True,
        truncation=True,
        max_length=512,
        add_special_tokens=True,
    )
    special_mask = tokenizer.get_special_tokens_mask(enc["input_ids"], already_has_special_tokens=True)
    labels = []
    cursor = 0
    for is_special in special_mask:
        if is_special == 1:
            labels.append(-100)
        else:
            if cursor < len(expanded_labels):
                labels.append(expanded_labels[cursor])
                cursor += 1
            else:
                labels.append(-100)
    return {**enc, "labels": labels}

# Xử lý từng mẫu (batched=False)
tokenized_ds = raw_ds.map(tokenize_and_align_labels, batched=False, remove_columns=["words", "ner_tags", "text"]) 

for split in tokenized_ds:
    print(split, tokenized_ds[split])


In [ ]:
# Khởi tạo model PhoBERT cho NER
from transformers import AutoConfig, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
import torch

# Fallback nếu MODEL_NAME chưa có
if 'MODEL_NAME' not in globals():
    MODEL_NAME = "vinai/phobert-base"

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

metric_name = "f1"
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    labels = p.label_ids
    true_labels = []
    pred_labels = []
    for pred, lab in zip(preds, labels):
        tl = []
        pl = []
        for p_i, l_i in zip(pred, lab):
            if l_i == -100:
                continue
            tl.append(id2label[l_i])
            pl.append(id2label[p_i])
        true_labels.append(tl)
        pred_labels.append(pl)
    return {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }

bs = 16
lr = 3e-5
epochs = 5
warmup_ratio = 0.1

args = TrainingArguments(
    output_dir="/kaggle/working/phobert-ner/checkpoints",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    save_total_limit=2,
    learning_rate=lr,
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs,
    num_train_epochs=epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    report_to=["none"],
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
# Đánh giá trên validation và test, lưu model ra /kaggle/working
print("Đánh giá trên validation...")
val_metrics = trainer.evaluate()
print(val_metrics)

print("Báo cáo chi tiết trên validation:")
val_preds = trainer.predict(tokenized_ds["validation"]).predictions
val_preds = np.argmax(val_preds, axis=-1)
val_labels = tokenized_ds["validation"]["labels"]

def decode_labels(preds, labels):
    decoded_true, decoded_pred = [], []
    for p, l in zip(preds, labels):
        tl, pl = [], []
        for p_i, l_i in zip(p, l):
            if l_i == -100:
                continue
            tl.append(id2label[int(l_i)])
            pl.append(id2label[int(p_i)])
        decoded_true.append(tl)
        decoded_pred.append(pl)
    return decoded_true, decoded_pred

val_true, val_pred = decode_labels(val_preds, val_labels)
print(classification_report(val_true, val_pred))

# Test
print("Đánh giá trên test...")
test_output = trainer.predict(tokenized_ds["test"])
print({
    "test_loss": float(test_output.metrics.get("test_loss", float("nan")))
})

# Lưu model và tokenizer
save_dir = "/kaggle/working/phobert-ner/best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

# Export hướng dẫn sử dụng inference
with open(os.path.join(save_dir, "README_infer.txt"), "w", encoding="utf-8") as f:
    f.write("Sử dụng PhoBERT NER đã fine-tune để gán nhãn TYPE/ATTR cho truy vấn.\n")

print("Đã lưu model tại:", save_dir)


In [ ]:
# Hàm suy luận cho truy vấn tự do
from transformers import pipeline

def build_ner_pipeline(model_dir: str):
    return pipeline(
        task="token-classification",
        model=model_dir,
        tokenizer=model_dir,
        aggregation_strategy="simple",  # gộp subwords
        device=0 if torch.cuda.is_available() else -1,
    )

infer = build_ner_pipeline("/kaggle/working/phobert-ner/best")

examples = [
    "Áo thun nam tay dài màu be phù hợp mùa đông giá khoảng 300 nghìn",
    "Váy maxi nữ màu trắng dáng rộng mặc mùa hè giá 800 nghìn",
]
for s in examples:
    print("\nQuery:", s)
    print(infer(s))


In [ ]:
# Suy luận trên /kaggle/input/abc/test.csv và lưu JSONL
import pandas as pd
import json
from transformers import pipeline

model_dir = "/kaggle/working/phobert-ner/best"
ner = pipeline(
    task="token-classification",
    model=model_dir,
    tokenizer=model_dir,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

def convert_numpy_types(obj):
    """Chuyển đổi numpy types thành Python types để JSON serializable"""
    if hasattr(obj, 'item'):  # numpy scalar
        return obj.item()
    elif isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    else:
        return obj

df_test = pd.read_csv('/kaggle/input/abc/test.csv')
outputs = []
for i, row in df_test.iterrows():
    text = str(row['text'])
    ents = ner(text)
    # Chuyển đổi numpy types để JSON serializable
    ents_clean = convert_numpy_types(ents)
    outputs.append({
        'id': int(i),
        'text': text,
        'entities': ents_clean,
    })

out_path = '/kaggle/working/test_predictions.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for item in outputs:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print('Đã ghi', len(outputs), 'dòng tới', out_path)

# Hiển thị kết quả mẫu
print("\n=== KẾT QUẢ NHẬN DIỆN THỰC THỂ ===")
for i, item in enumerate(outputs[:5]):  # Hiển thị 5 mẫu đầu
    print(f"\n--- Mẫu {i+1} ---")
    print(f"Text: {item['text']}")
    print("Entities:")
    for ent in item['entities']:
        print(f"  - {ent['word']} -> {ent['entity_group']} (confidence: {ent['score']:.3f})")
    
if len(outputs) > 5:
    print(f"\n... và {len(outputs) - 5} mẫu khác")
